# 01 — Récupération des données du réseau Sentinelles

Le réseau Sentinelles, ce sont des médecins généralistes volontaires qui déclarent chaque
semaine, depuis 1984, les cas vus pour un petit nombre de pathologies. L'Inserm en tire une
estimation d'incidence nationale et régionale, publiée en libre accès sur `sentiweb.fr`.

Je m'intéresse à l'indicateur **syndromes grippaux** (n° 3). L'export « all » donne tout
l'historique en un CSV ; le téléchargement et le parsing des semaines ISO sont dans
`src/data_loader.py`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import data_loader, preprocessing

In [2]:
brut = data_loader.load_incidence(3, "PAY")
print("période :", brut.index.min().date(), "->", brut.index.max().date())
print("nombre de semaines :", len(brut))
brut[["week", "inc", "inc100"]].head()

période : 1984-10-29 -> 2026-08-17
nombre de semaines : 2182


,week,inc,inc100
date,,,
1984-10-29,198444,68422,125
1984-11-05,198445,135223,246
1984-11-12,198446,87330,159
1984-11-19,198447,72029,131
1984-11-26,198448,78620,143


`inc` = nombre de cas estimé sur la France, `inc100` = taux pour 100 000 habitants. Je
travaillerai sur le **taux pour 100 000** : il reste comparable dans le temps malgré la
croissance de la population.

`serie_hebdo()` fait le nettoyage : passage en numérique, calendrier hebdomadaire strict,
interpolation des rares trous courts, et je coupe l'avant-1990 (méthode d'estimation moins
stable au tout début).

In [3]:
s = preprocessing.serie_hebdo(3, "PAY")
print(s.shape, "| valeurs manquantes :", int(s.isna().sum()))
s.describe().round(1)

(1881,) | valeurs manquantes : 0


count    1881.0
mean       90.2
std       162.4
min         0.0
25%         9.0
50%        28.0
75%        79.0
max      1565.0
Name: taux_100k, dtype: float64

In [4]:
# Vue régionale — disponible aussi, utile pour une extension mais pas pour le modèle principal.
reg = data_loader.load_incidence(3, "REG")
print("lignes :", len(reg), "| entités géo (régions, avant/après réforme 2016) :",
      reg["geo_name"].nunique())
reg.groupby("geo_name").size().sort_values(ascending=False).head()

lignes : 49874 | entités géo (régions, avant/après réforme 2016) : 23


geo_name
ALSACE             2182
AQUITAINE          2182
AUVERGNE           2182
BASSE-NORMANDIE    2182
BOURGOGNE          2182
dtype: int64

On garde la série nationale de ~1 880 semaines pour la suite. Le régional pourra servir à
tester si un modèle par région, agrégé, bat le modèle national — c'est noté comme
prolongement.